In [1]:
"""
Step 4c: Pooled event-dummy regression (Analysis 4, headline statistical test).

Answers the underlying question directly with ONE overall effect size and
p-value, instead of the per-event pre/post comparisons in analyze_event_study.py:
"across all 11 events pooled together, is burnout activity higher in the 30
days after an event than in the 30 days before -- controlling for day-of-week,
Patch Tuesday, each event's own baseline level, and daily Reddit post volume?"

Model: burnout_count ~ post_event_dummy + day_of_week + patch_tuesday
                        + event fixed effects
       with an offset of log(daily total post volume), so this estimates a
       RATE -- burnout posts as a share of all Reddit activity that day --
       rather than a raw count that would just track Reddit's overall growth
       from 2018 to 2026.

Uses a Poisson GLM by default, and automatically switches to Negative Binomial
if the data show overdispersion (very common with real count data -- Poisson
assumes mean == variance, which social-media volume data essentially never
satisfies; ignoring this understates your p-values).

Standard errors are clustered by event, since the ~60 days inside one event's
window aren't independent observations of "does a random day have more or less
burnout" -- they're correlated by construction.

Reads event_study_aligned.csv. No raw file access here.
"""

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import os

OUT_DIR = "/Users/nadia/Desktop/redditRun_june/event_study_v2/"
ALIGNED_CSV = os.path.join(OUT_DIR, "event_study_aligned.csv")
ALPHA = 0.05
OFFSET_COL = "n_posts_total"  # set to None to model raw counts instead of a rate

# raw count metrics to test -- NOT the _resid/_norm/_resid_z versions.
# Poisson/Negative Binomial models need actual counts, not deseasonalized
# z-scores (day-of-week and Patch Tuesday are handled as covariates here
# instead, which is the correct way to control for them in a count model).
METRIC_CANDIDATES = ["n_burnout", "n_EX", "n_EMO", "n_COG", "n_MD"]


def is_patch_tuesday(date):
    return 1 if (date.weekday() == 1 and 8 <= date.day <= 14) else 0


def run_pooled_regression(df, metric, offset_col):
    data = df.copy()
    data["post"] = (data["days_from_event"] >= 0).astype(int)
    data["date"] = pd.to_datetime(data["date"])
    data["day_of_week"] = data["date"].dt.dayofweek
    data["patch_tuesday"] = data["date"].apply(is_patch_tuesday)

    offset = None
    if offset_col is not None:
        if offset_col not in data.columns:
            print(f"  NOTE: offset column '{offset_col}' not found in aligned data -- "
                  f"running without an offset (results reflect raw counts, not a rate). "
                  f"Rerun build_event_study_data.py after this fix to get n_posts_total carried through.")
        elif not (data[offset_col] > 0).all():
            print(f"  NOTE: '{offset_col}' has zero/negative values on some days -- running without an offset")
        else:
            offset = np.log(data[offset_col])

    formula = f"{metric} ~ post + C(event_name) + C(day_of_week) + patch_tuesday"
    cluster_kwds = {"groups": data["event_name"]}

    poisson_fit = smf.glm(formula, data=data, family=sm.families.Poisson(), offset=offset).fit(
        cov_type="cluster", cov_kwds=cluster_kwds
    )
    dispersion = poisson_fit.pearson_chi2 / poisson_fit.df_resid
    overdispersed = dispersion > 1.5
    print(f"  Poisson dispersion ratio: {dispersion:.2f} "
          f"({'overdispersed -- switching to Negative Binomial' if overdispersed else 'looks OK, keeping Poisson'})")

    if overdispersed:
        try:
            fit = smf.glm(
                formula, data=data, family=sm.families.NegativeBinomial(alpha=1.0), offset=offset
            ).fit(cov_type="cluster", cov_kwds=cluster_kwds)
            model_used = "Negative Binomial (GLM)"
        except Exception as e:
            print(f"  NB fit failed ({e}) -- keeping Poisson result; interpret its p-value cautiously, "
                  f"it's likely too optimistic given the overdispersion")
            fit = poisson_fit
            model_used = "Poisson (NB fit failed, known overdispersed)"
    else:
        fit = poisson_fit
        model_used = "Poisson"

    return fit, model_used, dispersion


def main():
    if not os.path.exists(ALIGNED_CSV):
        print(f"Could not find {ALIGNED_CSV}. Run build_event_study_data.py first.")
        return

    df = pd.read_csv(ALIGNED_CSV)
    print(f"Loaded {len(df)} rows across {df['event_name'].nunique()} events")

    metrics = [m for m in METRIC_CANDIDATES if m in df.columns]
    if not metrics:
        print(f"None of {METRIC_CANDIDATES} found in {ALIGNED_CSV} -- check column names.")
        return
    print(f"Metrics to test: {metrics}")

    summary_rows = []
    for metric in metrics:
        print(f"\n{'=' * 70}\n{metric}\n{'=' * 70}")

        fit, model_used, dispersion = run_pooled_regression(df, metric, OFFSET_COL)

        coef = fit.params["post"]
        se = fit.bse["post"]
        pvalue = fit.pvalues["post"]
        irr = np.exp(coef)
        ci_low, ci_high = np.exp(coef - 1.96 * se), np.exp(coef + 1.96 * se)
        sig = "SIGNIFICANT" if pvalue < ALPHA else "not significant"

        print(f"\n  Model: {model_used}")
        print(f"  post-event effect: coef={coef:.4f}, p={pvalue:.4g} -> {sig} (alpha={ALPHA})")
        print(f"  Incidence Rate Ratio (IRR) = {irr:.3f}  (95% CI: {ci_low:.3f} - {ci_high:.3f})")
        if OFFSET_COL:
            pct_change = (irr - 1) * 100
            print(f"  Interpretation: burnout rate is {pct_change:+.1f}% in the 30 days after an event "
                  f"vs. the 30 days before, controlling for weekly pattern, Patch Tuesday, "
                  f"each event's own baseline, and total Reddit volume that day.")

        summary_path = os.path.join(OUT_DIR, f"pooled_regression_summary_{metric}.txt")
        with open(summary_path, "w") as f:
            f.write(f"Model used: {model_used}\nDispersion ratio: {dispersion:.2f}\n\n")
            f.write(str(fit.summary()))
        print(f"  Saved full model summary -> {summary_path}")

        summary_rows.append({
            "metric": metric, "model": model_used, "dispersion": round(dispersion, 3),
            "post_coef": round(coef, 4), "post_se": round(se, 4), "post_pvalue": round(pvalue, 6),
            "irr": round(irr, 3), "irr_ci_low": round(ci_low, 3), "irr_ci_high": round(ci_high, 3),
            "significant": pvalue < ALPHA,
        })

    summary_df = pd.DataFrame(summary_rows)
    out_path = os.path.join(OUT_DIR, "pooled_regression_summary.csv")
    summary_df.to_csv(out_path, index=False)
    print(f"\n{'=' * 70}\nFULL SUMMARY\n{'=' * 70}")
    print(summary_df.to_string(index=False))
    print(f"\nSaved -> {out_path}")

    print(f"\n{'=' * 70}\nHOW TO READ THIS\n{'=' * 70}")
    print("This is your single headline number per metric: the IRR tells you how much")
    print("higher (IRR > 1) or lower (IRR < 1) the burnout rate is after an event vs.")
    print("before, pooled across all 11 events, with day-of-week, Patch Tuesday, each")
    print("event's own baseline level, and total Reddit volume all controlled for.")
    print("\nThis is a stronger, more defensible test than the per-event Wilcoxon check")
    print("in analyze_event_study.py, because it uses every daily observation (not just")
    print("two collapsed pre/post means per event), properly separates each event's own")
    print("baseline from the shared 'post' effect via fixed effects, and clusters")
    print("standard errors by event so within-event day-to-day correlation doesn't")
    print("artificially inflate the significance.")
    print("\nIf dispersion was flagged as overdispersed and switched to Negative Binomial,")
    print("that's expected and good practice -- it means the Poisson model's assumption")
    print("(variance equals the mean) didn't hold, and ignoring that would have made the")
    print("p-value look better than it should.")


if __name__ == "__main__":
    main()

Loaded 671 rows across 11 events
Metrics to test: ['n_burnout', 'n_EX', 'n_EMO', 'n_COG', 'n_MD']

n_burnout
  NOTE: offset column 'n_posts_total' not found in aligned data -- running without an offset (results reflect raw counts, not a rate). Rerun build_event_study_data.py after this fix to get n_posts_total carried through.
  Poisson dispersion ratio: 1.11 (looks OK, keeping Poisson)

  Model: Poisson
  post-event effect: coef=0.0022, p=0.9674 -> not significant (alpha=0.05)
  Incidence Rate Ratio (IRR) = 1.002  (95% CI: 0.901 - 1.114)
  Interpretation: burnout rate is +0.2% in the 30 days after an event vs. the 30 days before, controlling for weekly pattern, Patch Tuesday, each event's own baseline, and total Reddit volume that day.
  Saved full model summary -> /Users/nadia/Desktop/redditRun_june/event_study_v2/pooled_regression_summary_n_burnout.txt

n_EX
  NOTE: offset column 'n_posts_total' not found in aligned data -- running without an offset (results reflect raw counts, not 